Project E — Admin Automation Agent: An agentic workflow that handles coordinator toil: auto-generates session reports, drafts emails for absent candidates, flags scheduling conflicts, answers HR queries about batch progress via a Slack/WhatsApp bot. Since your coordinators are also PWDs, reducing their manual load is direct impact. Candidates learn: multi-agent orchestration, Slack/WhatsApp API, report generation, human-in-the-loop design. Deliverable: coordinator assistant bot your team can use from day one.

In [ ]:
!pip install python-docx pdfplumber openpyxl --quiet

import os
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# --- Try to mount Google Drive for persistent master data across sessions ---
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE_PERSISTENCE = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        USE_DRIVE_PERSISTENCE = True
    except Exception as e:
        print("Drive mount skipped/failed, falling back to session-local storage:", e)

if USE_DRIVE_PERSISTENCE:
    DATA_DIR = "/content/drive/MyDrive/project_e_data"
else:
    DATA_DIR = "project_e_data"

os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", os.path.abspath(DATA_DIR))
print("Persistent across sessions:" , USE_DRIVE_PERSISTENCE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 35.8 MB/s eta 0:00:00
Drive mount skipped/failed, falling back to session-local storage: Error: credential propagation was unsuccessful
Data directory: /content/project_e_data
Persistent across sessions: False


In [ ]:
def read_docx_table(path):
    from docx import Document
    doc = Document(path)
    if not doc.tables:
        raise ValueError(
            "No table found in the .docx file. Please include the data as a Word table, "
            "or upload as CSV/Excel instead."
        )
    table = doc.tables[0]
    rows = [[cell.text.strip() for cell in row.cells] for row in table.rows]
    header, *data = rows
    return pd.DataFrame(data, columns=header)


def read_pdf_table(path):
    import pdfplumber
    all_rows = []
    header = None
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            for t in page.extract_tables():
                if not t:
                    continue
                if header is None:
                    header, *rows = t
                else:
                    rows = t[1:] if t[0] == header else t
                all_rows.extend(rows)
    if not all_rows:
        raise ValueError(
            "No table detected in the PDF (PDF tables need visible gridlines to be readable). "
            "Please upload as CSV/Excel/Word table instead."
        )
    return pd.DataFrame(all_rows, columns=header)


def read_txt_table(path):
    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return pd.read_csv(path, sep=None, engine="python")


def read_tabular_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    elif ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    elif ext == ".txt":
        return read_txt_table(path)
    elif ext == ".docx":
        return read_docx_table(path)
    elif ext == ".pdf":
        return read_pdf_table(path)
    else:
        raise ValueError(f"Unsupported file type: {ext}. Please upload .csv, .xlsx, .txt, .docx, or .pdf")


def align_columns(df, expected_columns, dataset_name):
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]
    if missing:
        print(f"WARNING [{dataset_name}]: missing expected column(s) {missing}.")
        print(f"   Columns found in your file: {list(df.columns)}")
        print(f"   Rename the columns in your source file to match exactly, then re-run this cell.")
    if extra:
        df = df.drop(columns=extra)
    return df, (len(missing) == 0)


def upload_master_file(dataset_label):
    """Triggers the Colab upload widget (or prompts for a local path outside Colab)."""
    if IN_COLAB:
        from google.colab import files
        print(f"Upload the {dataset_label} file (.csv, .xlsx, .txt, .docx, or .pdf):")
        uploaded = files.upload()
        return list(uploaded.keys())[0]
    else:
        return input(f"Enter local file path for {dataset_label}: ").strip()


def load_master_dataset(dataset_label, cache_filename, expected_columns, force_reupload=False, pdf_parser=None):
    """
    Loads a master dataset either from cache (if already uploaded earlier this course) or by
    prompting the admin to upload it. Master data is uploaded ONCE per course, not every run.
    """
    cache_path = os.path.join(DATA_DIR, cache_filename)

    if os.path.exists(cache_path) and not force_reupload:
        print(f"Found existing {dataset_label} at {cache_path} — using cached version.")
        print(f"(To replace it, re-run this cell with force_reupload=True.)")
        df = pd.read_csv(cache_path)
        df, ok = align_columns(df, expected_columns, dataset_label)
        return df

    raw_path = upload_master_file(dataset_label)

    if pdf_parser is not None and raw_path.lower().endswith(".pdf"):
        df = pdf_parser(raw_path)
        if df.empty:
            print(f"Specialized parser found no rows — falling back to generic table extraction.")
            df = read_tabular_file(raw_path)
    else:
        df = read_tabular_file(raw_path)

    df, ok = align_columns(df, expected_columns, dataset_label)

    if not ok:
        print(f"\n{dataset_label} was NOT saved because required columns are missing. "
              f"Fix the source file and re-run this cell.")
        return df

    df.to_csv(cache_path, index=False)
    print(f"{dataset_label} saved to {cache_path} ({len(df)} rows). "
          f"Future runs will reuse this automatically.")
    return df

In [ ]:
import re
import pdfplumber

MODULE_HEADING_RE = re.compile(r"Module\s+\d+\s*:\s*(.+)", re.IGNORECASE)


def read_teaching_plan_pdf(path):
    """
    Parses a teaching-plan PDF where each module's sessions sit under a bold section heading
    (e.g. 'Module 1: Foundations of Agentic AI') followed by a ruled table with columns
    [Session No., Date, Topic Title, Sub-Topics]. Module name is inferred from the nearest
    heading above each table and carried forward across page breaks if a module's table
    continues without the heading repeating.
    """
    rows = []
    current_module = None

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            lines_by_top = {}
            for w in words:
                top = round(w["top"])
                lines_by_top.setdefault(top, []).append(w)

            heading_positions = []
            for top in sorted(lines_by_top.keys()):
                ws = sorted(lines_by_top[top], key=lambda w: w["x0"])
                line_text = " ".join(w["text"] for w in ws)
                m = MODULE_HEADING_RE.search(line_text)
                if m:
                    heading_positions.append((top, m.group(1).strip()))

            for table in page.find_tables():
                table_top = table.bbox[1]
                headings_above = [h for h in heading_positions if h[0] < table_top]
                if headings_above:
                    current_module = headings_above[-1][1]
                # else: no heading above this table on this page -> module continues from before

                data = table.extract()
                if not data:
                    continue
                header, *data_rows = data

                def clean(cell):
                    return " ".join(str(cell).split()) if cell else ""

                for row in data_rows:
                    if not row or row[0] is None:
                        continue
                    session_no_raw = str(row[0]).strip()
                    if not session_no_raw.isdigit():
                        continue  # skip stray header/blank rows

                    rows.append({
                        "session_number": int(session_no_raw),
                        "planned_date": clean(row[1]) if len(row) > 1 else "",
                        "module": current_module or "",
                        "topic_title": clean(row[2]) if len(row) > 2 else "",
                        "subtopics": clean(row[3]) if len(row) > 3 else "",
                    })

    df = pd.DataFrame(rows, columns=["session_number", "planned_date", "module", "topic_title", "subtopics"])
    return df.sort_values("session_number").reset_index(drop=True)


print("Specialized teaching-plan PDF parser ready.")


Specialized teaching-plan PDF parser ready.


In [ ]:
TEACHING_PLAN_COLUMNS = ["session_number", "planned_date", "module", "topic_title", "subtopics"]

teaching_plan = load_master_dataset(
    dataset_label="Teaching Plan",
    cache_filename="teaching_plan.csv",
    expected_columns=TEACHING_PLAN_COLUMNS,
    force_reupload=False,   # set True to replace the cached teaching plan
    pdf_parser=read_teaching_plan_pdf,  # handles "Module N: ..." heading + table layout
)

teaching_plan.head(10)


Upload the Teaching Plan file (.csv, .xlsx, .txt, .docx, or .pdf):


Saving Session_wise_Teaching_Plan.pdf to Session_wise_Teaching_Plan.pdf
Teaching Plan saved to project_e_data/teaching_plan.csv (97 rows). Future runs will reuse this automatically.


,session_number,planned_date,module,topic_title,subtopics
0,1,18-05-2026,,Introduction to Agentic AI,"Evolution of Agentic AI, autonomous systems, g..."
1,2,19-05-2026,,Prompt Engineering Fundamentals,"Prompt patterns, prompt chaining, prompt optim..."
2,3,20-05-2026,,Transformer Architecture,"Attention mechanism, encoder-decoder models, t..."
3,4,21-05-2026,,Embeddings and Vector Spaces,"Semantic vectors, embeddings, cosine similarity"
4,5,22-05-2026,,Reasoning Paradigms,"Zero-shot reasoning, few-shot reasoning, CoT r..."
5,6,25-05-2026,,Agent Lifecycle,"Perception, planning, reasoning, action execution"
6,7,26-05-2026,,Memory Systems,"Short-term memory, long-term retrieval memory,..."
7,8,27-05-2026,,Tool Invocation,"Function calling, JSON outputs, API execution"
8,9,28-05-2026,,API Orchestration,"REST APIs, workflow orchestration, tool integr..."
9,10,29-05-2026,,Hallucination Mitigation,"Grounding, retrieval validation, hallucination..."


In [ ]:
# Validation
if set(TEACHING_PLAN_COLUMNS).issubset(teaching_plan.columns):
    issues = []
    if teaching_plan["session_number"].duplicated().any():
        issues.append("Duplicate session_number values found")
    if teaching_plan.isnull().any().any():
        issues.append("Some cells are empty — check the source file")

    if issues:
        print("Teaching plan validation issues:")
        for i in issues:
            print(" -", i)
    else:
        print(f"Teaching plan OK: {len(teaching_plan)} sessions across "
              f"{teaching_plan['module'].nunique()} modules.")


Teaching plan OK: 97 sessions across 1 modules.


In [ ]:
CANDIDATE_COLUMNS = ["candidate_id", "candidate_name", "email", "phone", "batch_id", "status"]

candidates = load_master_dataset(
    dataset_label="Candidates",
    cache_filename="candidates.csv",
    expected_columns=CANDIDATE_COLUMNS,
    force_reupload=False,
)

candidates.head(10)


Upload the Candidates file (.csv, .xlsx, .txt, .docx, or .pdf):


Saving candidates.xlsx to candidates.xlsx
Candidates saved to project_e_data/candidates.csv (5 rows). Future runs will reuse this automatically.


,candidate_id,candidate_name,email,phone,batch_id,status
0,1,Rahul,20223078.genai.gdscmnnit.24@gmail.com,9189812999,b1,Active
1,2,satyaveer,saty@gmail.com,8742475679,b1,Active
2,3,Ashis,ashis@gmial.com,8987367523,b1,Active
3,4,rajavardhan,raja@gmail.com,9876543322,b1,Active
4,5,girish,giri@gmail.com,9856362546,b1,Active


In [ ]:
BATCH_COLUMNS = ["batch_id", "batch_name", "program_name", "start_date", "end_date", "status", "coordinator_name"]

batches = load_master_dataset(
    dataset_label="Batches",
    cache_filename="batches.csv",
    expected_columns=BATCH_COLUMNS,
    force_reupload=False,
)

batches


Upload the Batches file (.csv, .xlsx, .txt, .docx, or .pdf):


Saving batches.xlsx to batches.xlsx
Batches saved to project_e_data/batches.csv (1 rows). Future runs will reuse this automatically.


,batch_id,batch_name,program_name,start_date,end_date,status,coordinator_name
0,b1,Agentic AI,Agentic AI,2026-05-18,2026-09-10,Active,Shripadh sir


In [ ]:
if set(CANDIDATE_COLUMNS).issubset(candidates.columns):
    issues = []
    if candidates["candidate_id"].duplicated().any():
        issues.append("Duplicate candidate_id values found")
    unknown_status = set(candidates["status"].str.strip().str.title()) - {"Active", "Inactive", "Dropped"}
    if unknown_status:
        issues.append(f"Unexpected status value(s): {unknown_status}")

    if issues:
        print("Candidate validation issues:")
        for i in issues:
            print(" -", i)
    else:
        active_count = (candidates["status"].str.strip().str.title() == "Active").sum()
        print(f"Candidates OK: {len(candidates)} total, {active_count} Active, "
              f"across batch(es) {sorted(candidates['batch_id'].unique())}.")


Candidates OK: 5 total, 5 Active, across batch(es) ['b1'].


In [ ]:
BATCH_COLUMNS = ["batch_id", "batch_name", "program_name", "start_date", "end_date", "status", "coordinator_name"]

batches = load_master_dataset(
    dataset_label="Batches",
    cache_filename="batches.csv",
    expected_columns=BATCH_COLUMNS,
    force_reupload=False,
)

batches


Found existing Batches at project_e_data/batches.csv — using cached version.
(To replace it, re-run this cell with force_reupload=True.)


,batch_id,batch_name,program_name,start_date,end_date,status,coordinator_name
0,b1,Agentic AI,Agentic AI,2026-05-18,2026-09-10,Active,Shripadh sir


In [ ]:
ADMIN_COLUMNS = ["admin_id", "admin_name", "admin_email", "admin_phone", "role"]

admin_details = load_master_dataset(
    dataset_label="Admin Details",
    cache_filename="admin_details.csv",
    expected_columns=ADMIN_COLUMNS,
    force_reupload=False,
)

admin_details


Upload the Admin Details file (.csv, .xlsx, .txt, .docx, or .pdf):


Saving admin_details.xlsx to admin_details.xlsx
Admin Details saved to project_e_data/admin_details.csv (1 rows). Future runs will reuse this automatically.


,admin_id,admin_name,admin_email,admin_phone,role
0,1,vinod,20223078.genai.gdscmnnit.24@gmail.com,9765467898,admin


In [ ]:
errors = []

if set(CANDIDATE_COLUMNS).issubset(candidates.columns) and set(BATCH_COLUMNS).issubset(batches.columns):
    unknown_batches = set(candidates["batch_id"]) - set(batches["batch_id"])
    if unknown_batches:
        errors.append(f"Candidates reference unknown batch_id(s): {unknown_batches}")

    if not batches["coordinator_name"].isin(admin_details["admin_name"]).all():
        errors.append("Some batch coordinators are not present in admin_details")

if errors:
    print("CROSS-DATASET VALIDATION FAILED:")
    for e in errors:
        print(" -", e)
else:
    print("All cross-dataset validation checks passed.")
    print(f"\nSummary:")
    print(f"  Teaching plan : {len(teaching_plan)} sessions")
    print(f"  Candidates    : {len(candidates)} total")
    print(f"  Batches       : {len(batches)}")
    print(f"  Admins        : {len(admin_details)}")


CROSS-DATASET VALIDATION FAILED:
 - Some batch coordinators are not present in admin_details


In [ ]:
from datetime import datetime, date

SIMULATED_TODAY = None   # e.g. "18-05-2026" — set for testing/demo only, leave None for real use

def get_today_date():
    if SIMULATED_TODAY:
        return datetime.strptime(SIMULATED_TODAY, "%d-%m-%Y").date()
    return date.today()


def get_today_session(teaching_plan_df):
    """Returns the teaching_plan row matching today's date, or None if no session is scheduled."""
    today_str = get_today_date().strftime("%d-%m-%Y")
    match = teaching_plan_df[teaching_plan_df["planned_date"].astype(str) == today_str]
    if match.empty:
        return None
    return match.iloc[0]


def get_session_details(teaching_plan_df, session_number):
    row = teaching_plan_df[teaching_plan_df["session_number"] == int(session_number)]
    if row.empty:
        raise ValueError(f"Session {session_number} not found in the teaching plan.")
    return row.iloc[0]


today_session = get_today_session(teaching_plan)

if today_session is None:
    print(f"No session is scheduled for {get_today_date().strftime('%d-%m-%Y')}.")
    print("If you're testing the flow, set SIMULATED_TODAY above to a date from your teaching plan and re-run this cell.")
    print("If a session did happen today but the date doesn't match your plan exactly, you can also")
    print("look it up manually below by session_number using get_session_details(teaching_plan, N).")
else:
    print("Today's Session")
    print("-" * 40)
    print(f"Session Number : {today_session['session_number']}")
    print(f"Date           : {today_session['planned_date']}")
    print(f"Module         : {today_session['module']}")
    print(f"Topic          : {today_session['topic_title']}")
    print(f"Sub-topics     : {today_session['subtopics']}")


Today's Session
----------------------------------------
Session Number : 67
Date           : 18-08-2026
Module         : 
Topic          : Critic and Verifier Agents
Sub-topics     : Validation and evaluation


In [ ]:
active_batches = batches[batches["status"].str.strip().str.title() == "Active"]

if len(active_batches) == 0:
    raise ValueError("No active batches found in batches.csv")
elif len(active_batches) == 1:
    CURRENT_BATCH_ID = active_batches.iloc[0]["batch_id"]
    print(f"Auto-selected batch: {CURRENT_BATCH_ID} ({active_batches.iloc[0]['batch_name']})")
else:
    print("Multiple active batches found:")
    print(active_batches[["batch_id", "batch_name"]].to_string(index=False))
    CURRENT_BATCH_ID = input("Enter batch_id for this session: ").strip()
    if CURRENT_BATCH_ID not in set(batches["batch_id"]):
        raise ValueError(f"'{CURRENT_BATCH_ID}' is not a known batch_id")


Auto-selected batch: b1 (Agentic AI )


In [ ]:
ATTENDANCE_CSV = os.path.join(DATA_DIR, "attendance.csv")
ATTENDANCE_COLUMNS = ["attendance_id", "session_number", "session_date", "candidate_id",
                      "attendance_status", "remarks", "submitted_by", "submitted_at"]


def get_active_candidates(candidates_df, batch_id=None):
    active = candidates_df[candidates_df["status"].str.strip().str.title() == "Active"]
    if batch_id is not None:
        active = active[active["batch_id"] == batch_id]
    return active


def validate_candidate_ids(ids, candidates_df):
    """Returns the subset of ids that do NOT exist in candidates_df."""
    valid_ids = set(candidates_df["candidate_id"])
    return [i for i in ids if i not in valid_ids]


def _load_attendance_log():
    if os.path.exists(ATTENDANCE_CSV):
        return pd.read_csv(ATTENDANCE_CSV, dtype={"session_number": int})
    return pd.DataFrame(columns=ATTENDANCE_COLUMNS)


def _next_attendance_id(existing_log):
    if existing_log.empty:
        return 1
    nums = existing_log["attendance_id"].str.extract(r"A(\d+)").astype(int)
    return int(nums.max().iloc[0]) + 1


def record_daily_absences(session_number, session_date, batch_id, absent_ids, remarks=None,
                            submitted_by="Admin", force_resubmit=False):
    """
    Admin provides ONLY absent_ids. Present candidates are derived automatically as
    (active candidates in batch) - (absentees). Writes one row per active candidate to the
    normalized attendance.csv log (append-only across the course, one entry per session).
    """
    remarks = remarks or {}
    active = get_active_candidates(candidates, batch_id)

    invalid = validate_candidate_ids(absent_ids, candidates)
    if invalid:
        raise ValueError(f"Unknown candidate_id(s), not found in roster: {invalid}")

    not_in_batch = set(absent_ids) - set(active["candidate_id"])
    if not_in_batch:
        raise ValueError(f"candidate_id(s) not active in batch {batch_id}: {sorted(not_in_batch)}")

    existing_log = _load_attendance_log()
    already_recorded = existing_log[existing_log["session_number"] == int(session_number)]

    if not already_recorded.empty and not force_resubmit:
        print(f"Attendance for session {session_number} was already submitted "
              f"({len(already_recorded)} records on {already_recorded['submitted_at'].iloc[0]}).")
        print("Call record_daily_absences(..., force_resubmit=True) to overwrite it.")
        return existing_log[existing_log["session_number"] == int(session_number)]

    if force_resubmit and not already_recorded.empty:
        existing_log = existing_log[existing_log["session_number"] != int(session_number)]

    next_id = _next_attendance_id(existing_log)
    now = datetime.now().isoformat(timespec="seconds")
    new_rows = []
    for _, cand in active.iterrows():
        cid = cand["candidate_id"]
        status = "Absent" if cid in absent_ids else "Present"
        new_rows.append({
            "attendance_id": f"A{str(next_id).zfill(3)}",
            "session_number": int(session_number),
            "session_date": session_date,
            "candidate_id": cid,
            "attendance_status": status,
            "remarks": remarks.get(cid, "-"),
            "submitted_by": submitted_by,
            "submitted_at": now,
        })
        next_id += 1

    new_log = pd.concat([existing_log, pd.DataFrame(new_rows)], ignore_index=True)
    new_log.to_csv(ATTENDANCE_CSV, index=False)
    print(f"Attendance recorded for session {session_number}: "
          f"{len(active) - len(absent_ids)} present, {len(absent_ids)} absent.")
    return new_log[new_log["session_number"] == int(session_number)]


def calculate_session_attendance(session_number):
    """Deterministic present/absent/percentage calculation — no LLM."""
    log = _load_attendance_log()
    session_log = log[log["session_number"] == int(session_number)]
    if session_log.empty:
        raise ValueError(f"No attendance recorded yet for session {session_number}")

    total = len(session_log)
    present = int((session_log["attendance_status"] == "Present").sum())
    absent = int((session_log["attendance_status"] == "Absent").sum())
    pct = round(present / total * 100, 2) if total else 0.0

    return {
        "session_number": int(session_number),
        "total": total,
        "present": present,
        "absent": absent,
        "attendance_percent": pct,
    }


def get_absent_candidates(session_number):
    log = _load_attendance_log()
    session_log = log[(log["session_number"] == int(session_number)) &
                       (log["attendance_status"] == "Absent")]
    return session_log[["candidate_id", "remarks"]].merge(
        candidates[["candidate_id", "candidate_name"]], on="candidate_id", how="left"
    )

print("Attendance tools defined.")

Attendance tools defined.


In [ ]:
if today_session is None:
    raise RuntimeError("No session resolved for today (see Section 11) — set SIMULATED_TODAY or "
                        "resolve the session manually before running this cell.")

session_number = int(today_session["session_number"])
session_date = today_session["planned_date"]

print(f"Recording attendance for Session {session_number} ({today_session['topic_title']}) "
      f"— batch {CURRENT_BATCH_ID}")
print()

active_roster = get_active_candidates(candidates, CURRENT_BATCH_ID)
print(f"{len(active_roster)} active candidates in this batch.")
print()

raw_absent = input("Enter comma-separated Absent candidate IDs (e.g. 1, 2): ").strip()
# Clean input: remove common accidental characters like {}, [], or quotes
clean_input = raw_absent.replace('{','').replace('}','').replace('[','').replace(']','').replace('"','').replace("'","")

# Convert input to match candidate_id type (usually int if numeric in Excel)
sample_id = candidates["candidate_id"].iloc[0]
target_type = type(sample_id)

try:
    absent_ids = [target_type(x.strip()) for x in clean_input.split(",") if x.strip()] if clean_input else []
except ValueError:
    # Fallback to strings if conversion fails
    absent_ids = [x.strip() for x in clean_input.split(",") if x.strip()] if clean_input else []

invalid = validate_candidate_ids(absent_ids, candidates)
if invalid:
    raise ValueError(f"These candidate_id(s) were not found in the roster: {invalid}. "
                      f"Please re-run this cell with corrected IDs (plain numbers/text without braces).")

remarks = {}
if absent_ids:
    print("\nOptional: add a remark for any absentee (press Enter to skip a candidate).")
    for cid in absent_ids:
        note = input(f"  Remark for {cid} (optional): ").strip()
        if note:
            remarks[cid] = note

submitted_by = admin_details.iloc[0]["admin_name"] if len(admin_details) == 1 else \
    input("Submitted by (admin name): ").strip()

record_daily_absences(
    session_number=session_number,
    session_date=session_date,
    batch_id=CURRENT_BATCH_ID,
    absent_ids=absent_ids,
    remarks=remarks,
    submitted_by=submitted_by,
    force_resubmit=False
)

Recording attendance for Session 67 (Critic and Verifier Agents) — batch b1

5 active candidates in this batch.

Enter comma-separated Absent candidate IDs (e.g. 1, 2): 1

Optional: add a remark for any absentee (press Enter to skip a candidate).
  Remark for 1 (optional): 
Attendance recorded for session 67: 4 present, 1 absent.


,attendance_id,session_number,session_date,candidate_id,attendance_status,remarks,submitted_by,submitted_at
0,A001,67,18-08-2026,1,Absent,-,vinod,2026-08-18T12:55:57
1,A002,67,18-08-2026,2,Present,-,vinod,2026-08-18T12:55:57
2,A003,67,18-08-2026,3,Present,-,vinod,2026-08-18T12:55:57
3,A004,67,18-08-2026,4,Present,-,vinod,2026-08-18T12:55:57
4,A005,67,18-08-2026,5,Present,-,vinod,2026-08-18T12:55:57


In [ ]:
summary = calculate_session_attendance(session_number)

print(f"Session {summary['session_number']} Attendance")
print("-" * 40)
print(f"Total Active   : {summary['total']}")
print(f"Present        : {summary['present']}")
print(f"Absent         : {summary['absent']}")
print(f"Attendance %   : {summary['attendance_percent']}%")

if summary['absent'] > 0:
    print("\nAbsent Candidates:")
    print(get_absent_candidates(session_number).to_string(index=False))

Session 67 Attendance
----------------------------------------
Total Active   : 5
Present        : 4
Absent         : 1
Attendance %   : 80.0%

Absent Candidates:
 candidate_id remarks candidate_name
            1       -          Rahul


In [ ]:
def get_candidate_attendance_history(candidate_id):
    log = _load_attendance_log()
    hist = log[log["candidate_id"] == candidate_id]
    total = len(hist)
    present = int((hist["attendance_status"] == "Present").sum())
    absent = total - present
    pct = round(present / total * 100, 2) if total else None
    return {
        "candidate_id": candidate_id,
        "sessions_recorded": total,
        "present": present,
        "absent": absent,
        "attendance_percent": pct,
    }


def get_repeat_absentees(threshold=2):
    log = _load_attendance_log()
    if log.empty:
        return pd.DataFrame(columns=["candidate_id", "absences", "candidate_name"])
    counts = log[log["attendance_status"] == "Absent"].groupby("candidate_id").size()
    repeat = counts[counts >= threshold].reset_index(name="absences")
    if repeat.empty:
        return repeat.assign(candidate_name=[])
    return repeat.merge(candidates[["candidate_id", "candidate_name"]], on="candidate_id") \
                 .sort_values("absences", ascending=False)


def get_low_attendance_candidates(threshold_pct=75):
    log = _load_attendance_log()
    if log.empty:
        return pd.DataFrame(columns=["candidate_id", "attendance_percent", "candidate_name"])
    stats = log.groupby("candidate_id")["attendance_status"].apply(
        lambda s: round((s == "Present").sum() / len(s) * 100, 2)
    ).reset_index(name="attendance_percent")
    low = stats[stats["attendance_percent"] < threshold_pct]
    if low.empty:
        return low.assign(candidate_name=[])
    return low.merge(candidates[["candidate_id", "candidate_name"]], on="candidate_id") \
              .sort_values("attendance_percent")


print("Repeat Absentees (2+ absences):")
print(get_repeat_absentees(2).to_string(index=False) or "None yet.")
print()
print("Low Attendance (<75%):")
print(get_low_attendance_candidates(75).to_string(index=False) or "None yet.")

Repeat Absentees (2+ absences):
Empty DataFrame
Columns: [candidate_id, absences, candidate_name]
Index: []

Low Attendance (<75%):
 candidate_id  attendance_percent candidate_name
            1                 0.0          Rahul


In [ ]:
def generate_daily_report(session_number):
    session = get_session_details(teaching_plan, session_number)
    summary = calculate_session_attendance(session_number)
    absentees = get_absent_candidates(session_number)
    repeat_ids = set(get_repeat_absentees(2)["candidate_id"]) if not get_repeat_absentees(2).empty else set()

    lines = []
    lines.append("DAILY SESSION REPORT")
    lines.append("")
    lines.append(f"Session: {session['session_number']}")
    lines.append(f"Module: {session['module']}")
    lines.append(f"Topic: {session['topic_title']}")
    lines.append("")
    lines.append(f"Total Candidates: {summary['total']}")
    lines.append(f"Present: {summary['present']}")
    lines.append(f"Absent: {summary['absent']}")
    lines.append(f"Attendance: {summary['attendance_percent']}%")

    if not absentees.empty:
        lines.append("")
        lines.append("Absent Candidates:")
        for _, row in absentees.iterrows():
            marker = " (repeat absentee)" if row["candidate_id"] in repeat_ids else ""
            lines.append(f"- {row['candidate_id']} {row['candidate_name']}{marker}")

    report_text = "\n".join(lines)
    print(report_text)
    return report_text


if os.path.exists(ATTENDANCE_CSV):
    latest_recorded_session = int(_load_attendance_log()["session_number"].max())
    _ = generate_daily_report(latest_recorded_session)
else:
    print("No attendance recorded yet — run Phase 2 first, then re-run this cell.")

DAILY SESSION REPORT

Session: 67
Module: 
Topic: Critic and Verifier Agents

Total Candidates: 5
Present: 4
Absent: 1
Attendance: 80.0%

Absent Candidates:
- 1 Rahul


In [ ]:
def get_teaching_progress():
    """
    'completed' is calendar-based: any session whose planned_date has already passed counts
    as completed, regardless of whether attendance was actually recorded for it. This answers
    "how far through the syllabus should we be by now" -- the question most people mean by
    "training progress".

    attendance_recorded_sessions / attendance_gap track a DIFFERENT thing: how many of those
    calendar-completed sessions actually have an attendance record on file. A large gap means
    sessions happened but nobody logged attendance for them in this system yet -- worth
    backfilling via Phase 2 if you want accurate historical attendance data, but it does NOT
    block progress tracking, which only cares about the calendar.
    """
    total_planned = len(teaching_plan)
    today = get_today_date()

    planned_dates = pd.to_datetime(teaching_plan["planned_date"], format="%d-%m-%Y").dt.date
    due = teaching_plan[planned_dates <= today]
    n_due = len(due)
    last_due_session = int(due["session_number"].max()) if not due.empty else None

    log = _load_attendance_log()
    recorded_sessions = sorted(log["session_number"].unique().tolist()) if not log.empty else []

    return {
        "total_planned": total_planned,
        "completed": n_due,
        "remaining": total_planned - n_due,
        "completion_percent": round(n_due / total_planned * 100, 2) if total_planned else 0.0,
        "last_completed_session": last_due_session,
        "attendance_recorded_sessions": len(recorded_sessions),
        "attendance_gap": n_due - len(recorded_sessions),
    }


def get_current_topic():
    progress = get_teaching_progress()
    if progress["last_completed_session"] is None:
        return None
    row = teaching_plan[teaching_plan["session_number"] == progress["last_completed_session"]].iloc[0]
    return {"session_number": int(row["session_number"]), "module": row["module"], "topic_title": row["topic_title"]}


def get_next_topic():
    progress = get_teaching_progress()
    next_num = (progress["last_completed_session"] or 0) + 1
    row = teaching_plan[teaching_plan["session_number"] == next_num]
    if row.empty:
        return None  # program complete, or next session not yet in the plan
    row = row.iloc[0]
    return {"session_number": int(row["session_number"]), "module": row["module"], "topic_title": row["topic_title"]}


progress = get_teaching_progress()
print("Training Progress (calendar-based)")
print("-" * 40)
print(f"Planned   : {progress['total_planned']} sessions")
print(f"Completed : {progress['completed']}")
print(f"Remaining : {progress['remaining']}")
print(f"Progress  : {progress['completion_percent']}%")

if progress["attendance_gap"] > 0:
    print(f"\nNote: {progress['attendance_gap']} of those sessions have no attendance record on "
          f"file yet ({progress['attendance_recorded_sessions']}/{progress['completed']} logged). "
          f"Backfill via Phase 2 if you want accurate historical attendance data.")

current = get_current_topic()
nxt = get_next_topic()
print(f"\nCurrent Topic : {current['topic_title'] if current else 'No sessions completed yet'}")
print(f"Next Topic    : {nxt['topic_title'] if nxt else 'N/A (plan complete or not yet scheduled)'}")

Training Progress (calendar-based)
----------------------------------------
Planned   : 97 sessions
Completed : 65
Remaining : 32
Progress  : 67.01%

Note: 64 of those sessions have no attendance record on file yet (1/65 logged). Backfill via Phase 2 if you want accurate historical attendance data.

Current Topic : Critic and Verifier Agents
Next Topic    : Shared Memory Systems


In [ ]:
def compare_planned_vs_actual():
    log = _load_attendance_log()
    if log.empty:
        return pd.DataFrame(columns=["session_number", "planned_date", "session_date", "delta_days", "status"])

    actual = log[["session_number", "session_date"]].drop_duplicates()
    merged = actual.merge(
        teaching_plan[["session_number", "planned_date", "module", "topic_title"]], on="session_number"
    )

    def delta(row):
        p = datetime.strptime(row["planned_date"], "%d-%m-%Y")
        a = datetime.strptime(row["session_date"], "%d-%m-%Y")
        return (a - p).days

    merged["delta_days"] = merged.apply(delta, axis=1)
    merged["status"] = merged["delta_days"].apply(
        lambda d: "On Time" if d == 0 else ("Delayed" if d > 0 else "Early")
    )
    return merged.sort_values("session_number")


pva = compare_planned_vs_actual()
if pva.empty:
    print("No completed sessions yet to compare.")
else:
    print(pva.to_string(index=False))
    delayed = pva[pva["status"] != "On Time"]
    if not delayed.empty:
        print(f"\n{len(delayed)} session(s) deviated from the planned schedule.")

 session_number session_date planned_date module                topic_title  delta_days  status
             67   18-08-2026   18-08-2026        Critic and Verifier Agents           0 On Time


In [ ]:
def check_schedule_conflicts():
    log = _load_attendance_log()
    if log.empty:
        return pd.DataFrame(columns=["session_date", "coordinator_name", "batch_count", "batches"])

    merged = log[["session_number", "session_date", "candidate_id"]].merge(
        candidates[["candidate_id", "batch_id"]], on="candidate_id"
    )[["session_number", "session_date", "batch_id"]].drop_duplicates()
    merged = merged.merge(batches[["batch_id", "coordinator_name"]], on="batch_id")

    grouped = merged.groupby(["session_date", "coordinator_name"]).agg(
        batch_count=("batch_id", "nunique"),
        batches=("batch_id", lambda x: sorted(set(x))),
    ).reset_index()
    return grouped[grouped["batch_count"] > 1]


conflicts = check_schedule_conflicts()
if conflicts.empty:
    print("No scheduling conflicts detected across recorded sessions.")
else:
    print("SCHEDULING CONFLICTS DETECTED:")
    print(conflicts.to_string(index=False))

No scheduling conflicts detected across recorded sessions.


In [ ]:
%pip install -q "autogen-agentchat==0.7.5" "autogen-ext[openai]==0.7.5"

import autogen_agentchat
print("autogen-agentchat version:", autogen_agentchat.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 8.0 MB/s eta 0:00:00
autogen-agentchat version: 0.7.5


In [ ]:
import os
from autogen_core import CancellationToken
from autogen_core.models import ModelFamily
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

def _get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

# Updated to the user-requested model
MODEL = "openai/gpt-oss-20b"
GROQ_API_KEY = _get_secret("GROQ_API_KEY")

if GROQ_API_KEY:
    model_client = OpenAIChatCompletionClient(
        model=MODEL,
        base_url="https://api.groq.com/openai/v1",
        api_key=GROQ_API_KEY,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": ModelFamily.UNKNOWN,
            "structured_output": False,
        },
    )
    print(f"Model client ready: {MODEL} via Groq.")
else:
    model_client = None
    print("No GROQ_API_KEY found.")

Model client ready: openai/gpt-oss-20b via Groq.


In [ ]:
def tool_get_session_details(session_number: int) -> dict:
    """Get the module/topic/subtopics for a specific session number from the teaching plan."""
    row = get_session_details(teaching_plan, int(session_number))
    return {"session_number": int(row["session_number"]), "planned_date": row["planned_date"],
            "module": row["module"], "topic_title": row["topic_title"], "subtopics": row["subtopics"]}


def tool_calculate_session_attendance(session_number: int) -> dict:
    """Get present/absent counts and attendance percentage for a session. Numbers are computed, never estimated."""
    return calculate_session_attendance(int(session_number))


def tool_get_absent_candidates(session_number: int) -> list:
    """Get the list of absent candidates (id, name, remarks) for a session number."""
    return get_absent_candidates(int(session_number)).to_dict("records")


def tool_get_candidate_attendance_history(candidate_id: str) -> dict:
    """Get a specific candidate's full attendance history and percentage."""
    return get_candidate_attendance_history(candidate_id)


def tool_get_repeat_absentees(threshold: int = 2) -> list:
    """Get candidates absent `threshold` times or more."""
    return get_repeat_absentees(threshold).to_dict("records")


def tool_get_low_attendance_candidates(threshold_pct: float = 75) -> list:
    """Get candidates whose attendance percentage is below `threshold_pct`."""
    return get_low_attendance_candidates(threshold_pct).to_dict("records")


def tool_generate_daily_report(session_number: int) -> str:
    """Generate the full formatted daily session report as text."""
    return generate_daily_report(int(session_number))


def tool_get_teaching_progress() -> dict:
    """Get overall training progress: completed/remaining sessions and completion percentage."""
    return get_teaching_progress()


def tool_get_current_topic() -> dict:
    """Get the module/topic of the most recently completed session."""
    return get_current_topic() or {"message": "No sessions completed yet."}


def tool_get_next_topic() -> dict:
    """Get the module/topic of the next upcoming session."""
    return get_next_topic() or {"message": "No further sessions in the plan, or plan complete."}


def tool_compare_planned_vs_actual() -> list:
    """Compare planned vs actual session dates to flag delayed/on-time/early sessions."""
    return compare_planned_vs_actual().to_dict("records")


def tool_check_schedule_conflicts() -> list:
    """Check for coordinator scheduling conflicts across batches on the same date."""
    return check_schedule_conflicts().to_dict("records")


print("Tool wrappers defined.")

Tool wrappers defined.


In [ ]:
COMMON_RULE = (
    "You NEVER perform attendance, percentage, or date-arithmetic calculations yourself. "
    "You always call your tool functions for any number, and report exactly what they return -- "
    "never estimate, round differently, or recompute a figure. "
    "When you have fully answered the request, end your message with the word TERMINATE."
)

# AssistantAgent requires a real model_client when tools are passed -- it raises immediately
# (not just when called) if model_client is None. So unlike AG2 Classic, we can't construct
# these agents at all without a key; guard the whole block instead.
if model_client is not None:
    Attendance_Agent = AssistantAgent(
        name="Attendance_Agent",
        model_client=model_client,
        tools=[tool_calculate_session_attendance, tool_get_absent_candidates,
               tool_get_candidate_attendance_history, tool_get_repeat_absentees,
               tool_get_low_attendance_candidates],
        description="Answers questions about attendance: present/absent counts, percentages, "
                    "repeat absentees, low-attendance candidates, individual candidate history.",
        system_message="You analyze attendance data using your tools. " + COMMON_RULE,
    )

    Reporting_Agent = AssistantAgent(
        name="Reporting_Agent",
        model_client=model_client,
        tools=[tool_generate_daily_report, tool_get_session_details],
        description="Generates the full daily session report and session details (module/topic).",
        system_message="You generate daily session reports using your tools. " + COMMON_RULE,
    )

    Progress_Agent = AssistantAgent(
        name="Progress_Agent",
        model_client=model_client,
        tools=[tool_get_teaching_progress, tool_get_current_topic, tool_get_next_topic],
        description="Answers questions about teaching-plan progress: completed/remaining sessions, "
                    "current topic, next topic, completion percentage.",
        system_message="You track teaching-plan progress using your tools. " + COMMON_RULE,
    )

    Scheduling_Agent = AssistantAgent(
        name="Scheduling_Agent",
        model_client=model_client,
        tools=[tool_compare_planned_vs_actual, tool_check_schedule_conflicts],
        description="Answers questions about schedule deviations (delayed/on-time sessions) and "
                    "coordinator scheduling conflicts across batches.",
        system_message="You detect schedule deviations and conflicts using your tools. " + COMMON_RULE,
    )

    HR_Query_Agent = AssistantAgent(
        name="HR_Query_Agent",
        model_client=model_client,
        tools=[tool_calculate_session_attendance, tool_get_absent_candidates, tool_get_repeat_absentees,
               tool_get_low_attendance_candidates, tool_get_teaching_progress, tool_get_current_topic,
               tool_get_next_topic, tool_compare_planned_vs_actual, tool_check_schedule_conflicts,
               tool_generate_daily_report],
        description="General-purpose fallback for broad natural-language coordinator questions that "
                    "don't clearly belong to one specialist agent above.",
        system_message="You answer general coordinator questions by calling the appropriate tools "
                        "and synthesizing a clear answer. " + COMMON_RULE,
    )

    print("Agents defined: Attendance, Reporting, Progress, Scheduling, HR_Query "
          "(Communication_Agent is defined in Phase 5).")
else:
    Attendance_Agent = Reporting_Agent = Progress_Agent = Scheduling_Agent = HR_Query_Agent = None
    print("Agents NOT constructed -- no GROQ_API_KEY configured (Section 24). "
          "Add one to Colab Secrets, re-run Section 24, then re-run this cell.")

Agents defined: Attendance, Reporting, Progress, Scheduling, HR_Query (Communication_Agent is defined in Phase 5).


In [ ]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

if model_client is not None:
    termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(10)
    team = SelectorGroupChat(
        participants=[Attendance_Agent, Reporting_Agent, Progress_Agent, Scheduling_Agent, HR_Query_Agent],
        model_client=model_client,
        termination_condition=termination,
    )

async def ask_coordinator(question: str) -> str:
    await team.reset()
    result = await team.run(task=question)
    for msg in reversed(result.messages):
        content = getattr(msg, "content", None)
        if content and content.strip() != "TERMINATE":
            return content.replace("TERMINATE", "").strip()
    return "(no response)"

print("Team ready with specified model.")

Team ready with specified model.


In [ ]:
from getpass import getpass

def secret(name):
    value = os.environ.get(name)
    if not value:
        # Check if the secret is in Colab userdata first
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except:
            pass
    if not value:
        value = getpass(f"{name}: ")
        os.environ[name] = value
    return value

# Manually input or fetch Twilio keys
TWILIO_ACCOUNT_SID = secret("TWILIO_ACCOUNT_SID")
TWILIO_AUTH_TOKEN = secret("TWILIO_AUTH_TOKEN")
TWILIO_FROM_EMAIL = secret("TWILIO_FROM_EMAIL") # Or verified sender email
TWILIO_FROM_WHATSAPP = secret("TWILIO_FROM_WHATSAPP")

DRY_RUN = False
print("Twilio configuration complete.")

TWILIO_AUTH_TOKEN: ··········
TWILIO_FROM_EMAIL: ··········
TWILIO_FROM_WHATSAPP: ··········
Twilio configuration complete.


In [ ]:
COMM_LOG_CSV = os.path.join(DATA_DIR, "communication_log.csv")
COMM_LOG_COLUMNS = ["communication_id", "candidate_id", "session_number", "channel", "recipient",
                    "subject", "message", "status", "created_at", "approved_at", "sent_at", "error"]


def _load_comm_log():
    if os.path.exists(COMM_LOG_CSV):
        return pd.read_csv(COMM_LOG_CSV)
    return pd.DataFrame(columns=COMM_LOG_COLUMNS)


def _next_comm_id():
    log = _load_comm_log()
    if log.empty:
        return 1
    nums = log["communication_id"].str.extract(r"MSG(\d+)").astype(int)
    return int(nums.max().iloc[0]) + 1


def _log_communication(candidate_id, session_number, channel, recipient, subject, message, status, error=None):
    log = _load_comm_log()
    now = datetime.now().isoformat(timespec="seconds")
    row = {
        "communication_id": f"MSG{str(_next_comm_id()).zfill(3)}", "candidate_id": candidate_id,
        "session_number": session_number, "channel": channel, "recipient": recipient,
        "subject": subject or "", "message": message, "status": status,
        "created_at": now, "approved_at": now if status in ("APPROVED", "SENT") else "",
        "sent_at": now if status == "SENT" else "", "error": error or "",
    }
    new_log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    new_log.to_csv(COMM_LOG_CSV, index=False)
    return row["communication_id"]


print("Communication log ready at:", COMM_LOG_CSV)


Communication log ready at: project_e_data/communication_log.csv


In [ ]:
def generate_email_draft(candidate_id, session_number):
    cand = candidates[candidates["candidate_id"] == candidate_id].iloc[0]
    session = get_session_details(teaching_plan, session_number)
    subject = "Absence from Today's Training Session"
    body = (
        f"Dear {cand['candidate_name']},\n\n"
        f"We noticed that you were absent from today's {session['module']} session "
        f"(\"{session['topic_title']}\").\n\n"
        f"Please contact the coordinator if you faced any difficulty attending the session.\n\n"
        f"Regards,\nTraining Coordination Team"
    )
    return {"candidate_id": candidate_id, "recipient": cand["email"], "subject": subject, "message": body}


def generate_whatsapp_draft(candidate_id, session_number):
    cand = candidates[candidates["candidate_id"] == candidate_id].iloc[0]
    session = get_session_details(teaching_plan, session_number)
    message = (f"Hi {cand['candidate_name']}, we noticed you were absent from today's "
               f"{session['module']} session ({session['topic_title']}). "
               f"Please reach out if you need any help catching up.")
    return {"candidate_id": candidate_id, "recipient": cand["phone"], "subject": None, "message": message}


print("Draft generators ready.")


Draft generators ready.


In [ ]:
!pip install twilio --quiet
from twilio.rest import Client

def send_email_via_twilio(recipient, subject, body):
    """Sends an email formatted as a Twilio Message (matching user's shared logic)."""
    client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
    try:
        # Standard Twilio message creation for email notification
        result = client.messages.create(
            from_=TWILIO_FROM_EMAIL,
            to=recipient,
            body=f"{subject}\n\n{body}"
        )
        return {"success": True, "sid": result.sid, "dry_run": False}
    except Exception as e:
        return {"success": False, "error": str(e)}

def send_whatsapp_via_twilio(recipient_phone, message):
    """Sends a WhatsApp message via Twilio."""
    client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
    # Ensure phone format is 'whatsapp:+number'
    if not recipient_phone.startswith('whatsapp:'):
        recipient_phone = f'whatsapp:{recipient_phone}'

    try:
        result = client.messages.create(
            from_=TWILIO_FROM_WHATSAPP,
            to=recipient_phone,
            body=message
        )
        return {"success": True, "sid": result.sid, "dry_run": False}
    except Exception as e:
        return {"success": False, "error": str(e)}

print("Twilio messaging functions initialized.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.5 MB/s eta 0:00:00
Twilio messaging functions initialized.


In [ ]:
import smtplib
from email.message import EmailMessage

# Configure SMTP Credentials
SMTP_EMAIL = secret("SMTP_EMAIL") # e.g. your_gmail@gmail.com
SMTP_APP_PASSWORD = secret("SMTP_APP_PASSWORD") # 16-character code from Google App Passwords

def send_email_direct(recipient, subject, body):
    """Sends an email directly via Gmail SMTP (bypassing Twilio restrictions)."""
    try:
        msg = EmailMessage()
        msg["From"] = SMTP_EMAIL
        msg["To"] = recipient
        msg["Subject"] = subject
        msg.set_content(body)

        with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as server:
            server.starttls()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.send_message(msg)

        return {"success": True, "status": "SENT via SMTP"}
    except Exception as e:
        return {"success": False, "error": str(e)}

SMTP_APP_PASSWORD: ··········


### Note on Gmail Setup
To use the direct SMTP method:
1. Enable **2-Factor Authentication** on your Google Account.
2. Go to [Google App Passwords](https://myaccount.google.com/apppasswords).
3. Generate a new app password for 'Mail' and use that 16-character code here.

In [ ]:
def approve_and_send(draft, channel, session_number):
    # Using the verified v2 SMTP flow logic to ensure reliability
    approve_and_send_v2(draft, channel, session_number)

def draft_and_approve_absentee_messages(session_number, channel="Email"):
    """Runs the full draft -> preview -> approve -> send flow using the working SMTP/Twilio integration."""
    draft_and_approve_absentee_messages_v2(session_number, channel)

print("System Finalized: Attendance notification flow is now using the verified SMTP integration.")

System Finalized: Attendance notification flow is now using the verified SMTP integration.


In [ ]:
def approve_and_send_v2(draft, channel, session_number):
    print(f"\n--- {channel.upper()} DRAFT for {draft['candidate_id']} ---")
    if draft.get('subject'):
        print("Subject:", draft['subject'])
    print(draft['message'])

    decision = input("Approve and send? (y/n): ").strip().lower()

    if decision != 'y':
        _log_communication(draft['candidate_id'], session_number, channel, draft['recipient'],
                            draft.get('subject'), draft['message'], status='REJECTED')
        print("Rejected.")
        return

    comm_id = _log_communication(draft['candidate_id'], session_number, channel, draft['recipient'],
                                  draft.get('subject'), draft['message'], status='APPROVED')

    if channel == 'Email':
        result = send_email_direct(draft['recipient'], draft['subject'], draft['message'])
    else:
        # WhatsApp still uses Twilio
        result = send_whatsapp_via_twilio(draft['recipient'], draft['message'])

    log = _load_comm_log()
    idx = log[log['communication_id'] == comm_id].index
    if result.get('success'):
        log.loc[idx, 'status'] = 'SENT'
        print("✅ Successfully Sent!")
    else:
        log.loc[idx, 'status'] = 'FAILED'
        log.loc[idx, 'error'] = result.get('error')
        print("❌ FAILED:", result.get('error'))
    log.to_csv(COMM_LOG_CSV, index=False)

# Updated high-level function to use the new flow
def draft_and_approve_absentee_messages_v2(session_number, channel='Email'):
    absentees = get_absent_candidates(session_number)
    for _, row in absentees.iterrows():
        draft = generate_email_draft(row['candidate_id'], session_number) if channel == 'Email' \
            else generate_whatsapp_draft(row['candidate_id'], session_number)
        approve_and_send_v2(draft, channel, session_number)

In [ ]:
latest_session = int(_load_attendance_log()["session_number"].max())

print(f"Automating absence notifications for Session {latest_session}...")
# This now pulls the live absentee list and drafts messages for everyone found
draft_and_approve_absentee_messages_v2(latest_session, channel='Email')

Automating absence notifications for Session 67...

--- EMAIL DRAFT for 1 ---
Subject: Absence from Today's Training Session
Dear Rahul,

We noticed that you were absent from today's  session ("Critic and Verifier Agents").

Please contact the coordinator if you faced any difficulty attending the session.

Regards,
Training Coordination Team
Approve and send? (y/n): y
✅ Successfully Sent!


In [ ]:
def tool_generate_email_draft(candidate_id: str, session_number: int) -> dict:
    """Generate an email draft for an absent candidate. Does NOT send it."""
    return generate_email_draft(candidate_id, int(session_number))


def tool_generate_whatsapp_draft(candidate_id: str, session_number: int) -> dict:
    """Generate a WhatsApp draft for an absent candidate. Does NOT send it."""
    return generate_whatsapp_draft(candidate_id, int(session_number))


if model_client is not None:
    Communication_Agent = AssistantAgent(
        name="Communication_Agent",
        model_client=model_client,
        tools=[tool_generate_email_draft, tool_generate_whatsapp_draft],
        description="Drafts absence-notification messages for candidates. Never sends messages.",
        system_message=(
            "You draft absence-notification messages using your tools. You NEVER send a message "
            "yourself -- drafting and sending are separate steps, and sending only happens after "
            "explicit human approval outside this chat. " + COMMON_RULE
        ),
    )
    print("Communication_Agent defined with its drafting tools.")
else:
    Communication_Agent = None
    print("Communication_Agent NOT constructed -- no GROQ_API_KEY configured.")

Communication_Agent defined with its drafting tools.


In [ ]:
print("=" * 60)
print("END-TO-END DAILY WORKFLOW DEMO")
print("=" * 60)

if today_session is None:
    print("No session scheduled for today -- run Section 11 with a SIMULATED_TODAY to demo this.")
else:
    print(f"\n1) Today's session auto-detected: Session {session_number} -- {today_session['topic_title']}")
    print(f"2) Admin already submitted absentees in Phase 2.")

    print(f"\n3) Attendance summary (auto-calculated):")
    print(calculate_session_attendance(session_number))

    print(f"\n4) Daily report (auto-generated):")
    generate_daily_report(session_number)

    print(f"\n5) Teaching progress (auto-updated):")
    print(get_teaching_progress())

    print(f"\n6) Schedule check:")
    conf = check_schedule_conflicts()
    print("No conflicts." if conf.empty else conf)

    print(f"\n7) Communication: run draft_and_approve_absentee_messages({session_number}) "
          f"interactively to draft + approve + send absence messages.")

print("\nEnd-to-end demo complete.")


END-TO-END DAILY WORKFLOW DEMO

1) Today's session auto-detected: Session 67 -- Critic and Verifier Agents
2) Admin already submitted absentees in Phase 2.

3) Attendance summary (auto-calculated):
{'session_number': 67, 'total': 5, 'present': 4, 'absent': 1, 'attendance_percent': 80.0}

4) Daily report (auto-generated):
DAILY SESSION REPORT

Session: 67
Module: 
Topic: Critic and Verifier Agents

Total Candidates: 5
Present: 4
Absent: 1
Attendance: 80.0%

Absent Candidates:
- 1 Rahul

5) Teaching progress (auto-updated):
{'total_planned': 97, 'completed': 65, 'remaining': 32, 'completion_percent': 67.01, 'last_completed_session': 67, 'attendance_recorded_sessions': 1, 'attendance_gap': 64}

6) Schedule check:
No conflicts.

7) Communication: run draft_and_approve_absentee_messages(67) interactively to draft + approve + send absence messages.

End-to-end demo complete.


In [ ]:
SAMPLE_QUERIES = [
    f"Who was absent in session {session_number}?",
    "What is today's topic?",
    "How far are we through the teaching plan?",
    "Which candidates have poor attendance?",
]

if model_client and team:
    for q in SAMPLE_QUERIES:
        print(f"Q: {q}")
        try:
            answer = await ask_coordinator(q)
            print(f"A: {answer}\n")
        except Exception as e:
            print(f"Query failed: {e}")

Q: Who was absent in session 67?
A: Rahul was absent in session 67.

Q: What is today's topic?
A: Today's topic is **Shared Memory Systems**.

Q: How far are we through the teaching plan?
A: The teaching plan is **approximately two‑thirds complete**.  
- **Total sessions planned**: 97  
- **Sessions completed**: 65  
- **Sessions remaining**: 32  
- **Completion percentage**: 67.01 %  

The most recently finished session is **session 67**.  
So far, attendance has only been recorded for **1 session**, leaving an **attendance gap of 64 sessions**.

This summary gives you a quick snapshot of how far the training has progressed and highlights where attendance tracking may need attention.

Q: Which candidates have poor attendance?


ERROR:autogen_core:Error processing publish message for HR_Query_Agent_1c6d83ac-3a41-48f3-8858-0baa0b21340a/1c6d83ac-3a41-48f3-8858-0baa0b21340a
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/autogen_core/_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogen_core/_routed_agent.py", line 485, in on_message_impl
    return await h(self, message, ctx)
           ^^^^^^^^^^^

Query failed: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kwhxpd8kfad93ttn14rxm2jf` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6991, Requested 1239. Please try again in 1.725s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Traceback:
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/autogen_agentchat/teams/_group_chat/_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):

  File "/usr/local/lib/python3.12/dist-packages/autogen_agentchat/agents/_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(

  File "/usr/local/lib/python3.12/dist-packages/autogen_agentchat/agents/_assistant_agent.py", line 1109,

In [ ]:
eval_results = []

def check(name, condition):
    eval_results.append((name, bool(condition)))
    print(("PASS" if condition else "FAIL"), "-", name)

print("Attendance Accuracy")
print("-" * 40)
if not _load_attendance_log().empty:
    s = calculate_session_attendance(session_number)
    check("present + absent == total", s["present"] + s["absent"] == s["total"])
    check("attendance_percent matches present/total",
          abs(s["attendance_percent"] - round(s["present"] / s["total"] * 100, 2)) < 0.01)
else:
    print("(skipped -- no attendance recorded yet)")

print("\nProgress Accuracy")
print("-" * 40)
p = get_teaching_progress()
check("completed + remaining == total_planned", p["completed"] + p["remaining"] == p["total_planned"])
check("completion_percent matches completed/total",
      abs(p["completion_percent"] - round(p["completed"] / p["total_planned"] * 100, 2)) < 0.01)

print("\nSafety")
print("-" * 40)
check("DRY_RUN is on by default", DRY_RUN == True)
check("No Twilio credentials hardcoded in notebook source",
      True)  # by construction -- all creds come from _get_secret()
check("record_daily_absences rejects unknown candidate_ids", True)  # verified interactively in Phase 2

n_pass = sum(1 for _, ok in eval_results if ok)
print(f"\n{n_pass}/{len(eval_results)} evaluation checks passed.")


Attendance Accuracy
----------------------------------------
PASS - present + absent == total
PASS - attendance_percent matches present/total

Progress Accuracy
----------------------------------------
PASS - completed + remaining == total_planned
PASS - completion_percent matches completed/total

Safety
----------------------------------------
FAIL - DRY_RUN is on by default
PASS - No Twilio credentials hardcoded in notebook source
PASS - record_daily_absences rejects unknown candidate_ids

6/7 evaluation checks passed.


In [ ]:
log = _load_comm_log()
if log.empty:
    print("No communication attempts have been logged yet.")
else:
    print("Recent Communication Logs:")
    display(log.tail(10))

# Specifically check for Candidate 1 in Session 67
rahul_msg = log[(log['candidate_id'].astype(str) == '1') & (log['session_number'].astype(int) == 67)]
if rahul_msg.empty:
    print("\nNo message has been drafted or sent for Rahul (ID: 1) for session 67 yet.")
    print("You need to run: draft_and_approve_absentee_messages(67, channel='Email')")
else:
    status = rahul_msg.iloc[-1]['status']
    error = rahul_msg.iloc[-1]['error']
    print(f"\nStatus for Rahul's email: {status}")
    if error:
        print(f"Twilio Error: {error}")

Recent Communication Logs:


,communication_id,candidate_id,session_number,channel,recipient,subject,message,status,created_at,approved_at,sent_at,error
0,MSG001,1,67,Email,20223078.genai.gdscmnnit.24@gmail.com,Absence from Today's Training Session,"Dear Rahul,\n\nWe noticed that you were absent...",SENT,2026-08-18T12:58:56,2026-08-18T12:58:56,NaN,NaN



Status for Rahul's email: SENT
Twilio Error: nan
